In [91]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline




In [92]:
london = pd.read_csv("data/london.csv")
paris = pd.read_csv("data/paris.csv")
amsterdam = pd.read_csv("data/amsterdam.csv")
new_york = pd.read_csv("data/newyork.csv")


In [93]:
# did,t used merge as merge is used for stacking on basis of column name
# pd.merge(london,paris,amsterdam,new_york,on="city")
#we will use concat
#ignore_index=True tells Pandas to create a new continuous index after concatenating.
df = pd.concat(
    [london, paris, amsterdam, new_york],
    ignore_index=True
)

In [94]:
median_occupancy = df["estimated_occupancy_l365d"].median()

df["high_occupancy"] = (
    df["estimated_occupancy_l365d"] >= median_occupancy
).astype(int)

print(df["high_occupancy"].value_counts())


high_occupancy
1    70421
0    67797
Name: count, dtype: int64


In [95]:
features = [
    "city",
    "neighbourhood_cleansed",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "longitude",
    "latitude",
    "price",
    "number_of_reviews"
]

In [96]:
X = df[features]
y = df["high_occupancy"]

In [97]:
X.info()
y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138218 entries, 0 to 138217
Data columns (total 11 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   city                    138218 non-null  object 
 1   neighbourhood_cleansed  138218 non-null  object 
 2   room_type               138218 non-null  object 
 3   accommodates            138218 non-null  int64  
 4   bathrooms               138218 non-null  float64
 5   bedrooms                138218 non-null  float64
 6   beds                    138218 non-null  float64
 7   longitude               138218 non-null  float64
 8   latitude                138218 non-null  float64
 9   price                   138218 non-null  float64
 10  number_of_reviews       138218 non-null  int64  
dtypes: float64(6), int64(2), object(3)
memory usage: 11.6+ MB
<class 'pandas.core.series.Series'>
RangeIndex: 138218 entries, 0 to 138217
Series name: high_occupancy
Non-Null Count   Dty

In [98]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0,stratify=y)

In [99]:
# syntax
# preprocessor = ColumnTransformer(
#     transformers=[
#         (
#             "name",
#             transformer,
#             columns
#         ),
#         (
#             "name2",
#             transformer2,
#             columns2
#         )
#     ],
#     remainder="drop"   # optional
# )

#OneHotEncoder
# handle_unknown="ignore"
# This prevents errors if the model encounters a category during prediction that wasn't present during training.

preprocessor=ColumnTransformer(
                transformers=[
                    ("num",StandardScaler(),["bathrooms","bedrooms","beds","accommodates","longitude",
    "price",
    "number_of_reviews",
    "latitude"]),
                    ("cat",OneHotEncoder(handle_unknown="ignore"),["city","neighbourhood_cleansed","room_type"])
                             ])


In [100]:


pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("lr", LogisticRegression(max_iter=1000))
])


In [101]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bathrooms', 'bedrooms',
                                                   'beds', 'accommodates',
                                                   'longitude', 'price',
                                                   'number_of_reviews',
                                                   'latitude']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city',
                                                   'neighbourhood_cleansed',
                                                   'room_type'])])),
                ('lr', LogisticRegression(max_iter=1000))])

In [102]:
y_pred = pipeline.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7642526407176964
              precision    recall  f1-score   support

           0       0.71      0.87      0.78     13560
           1       0.84      0.66      0.74     14084

    accuracy                           0.76     27644
   macro avg       0.78      0.77      0.76     27644
weighted avg       0.78      0.76      0.76     27644

[[11798  1762]
 [ 4755  9329]]


In [103]:
# | Step                       |       MAE |       RMSE |          R² |
# | -------------------------- | --------: | ---------: | ----------: |
# | Original Linear Regression |     87.93 |     129.85 |       0.451 |
# | + Log(price)               |     78.23 |     123.86 |       0.500 |
# | + Latitude & Longitude     | **74.74** | **119.12** | **0.538** ✅ |

